## Notebook 02 — Fine-tune Student Model

Experiments 3 and 4 | Google Colab → Runtime > Change runtime type > T4 GPU.

| Experiment | Updated training data | `data.mode` | Saved model adapter |
|---|---|---|---|
| Exp 3 | D1: 469 train / 53 validation (522 total) | `distillation` | `updated_data/checkpoints/distillation_updated/` |
| Exp 4 | D2: 367 train / 41 validation (408 total) | `filtered` | `updated_data/checkpoints/filtered_updated/` |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import subprocess
from pathlib import Path
from google.colab import userdata

# Hugging Face token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add HF_TOKEN under Colab Secrets before running.')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token

repo = Path('/content/SC4001-Group133')
repo_url = 'https://github.com/TIFFMUN/SC4001-Group133.git'

if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', repo_url, str(repo)], check=True)
else:
    print('Using the existing repository in this Colab runtime.')

os.chdir(repo)
%pip install -q -r requirements.txt
%pip install -q -e .

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sc4001-group133 (pyproject.toml) ... done


In [3]:
# Original cache setup — run before importing Hugging Face or PyTorch models
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

# Keep model downloads in this Colab runtime; trained adapters stay in Drive.
# This also avoids reusing a possibly damaged previous model-download cache.
cache_dir = '/content/sc4001_hf_cache'
os.environ['HF_HOME'] = cache_dir
os.environ['HF_HUB_CACHE'] = f'{cache_dir}/hub'
os.environ['HF_XET_CACHE'] = f'{cache_dir}/xet'
os.environ.pop('TRANSFORMERS_CACHE', None)
print('Hugging Face cache:', os.environ['HF_HUB_CACHE'])

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001
Hugging Face cache: /content/sc4001_hf_cache/hub


In [4]:
# Link Notebook 01's updated data to the project (do not alter original Drive data)
from pathlib import Path
import shutil
from datasets import load_from_disk

project = Path('/content/SC4001-Group133')
drive_root = Path('/content/drive/MyDrive/SC4001')
original_data = drive_root / 'data'
updated_data = drive_root / 'updated_data'

# Confirm both updated datasets are already present.
for name in ('all', 'filtered'):
    folder = updated_data / 'processed' / name
    if not (folder / 'dataset_dict.json').is_file():
        raise FileNotFoundError(f'Missing updated {name} dataset: {folder}')

local_data = project / 'data'
if local_data.is_symlink():
    if local_data.resolve() not in (original_data.resolve(), updated_data.resolve()):
        raise RuntimeError(f'Unexpected existing data link: {local_data.resolve()}')
    if local_data.resolve() != updated_data.resolve():
        local_data.unlink()
elif local_data.exists():
    backup = project / 'data_repo_backup_notebook02'
    if backup.exists():
        raise RuntimeError('Local data/ exists and its backup name is already taken. Inspect before proceeding.')
    shutil.move(str(local_data), str(backup))
    print('Preserved existing local data folder at:', backup)

if not local_data.is_symlink():
    local_data.symlink_to(updated_data, target_is_directory=True)

assert local_data.resolve() == updated_data.resolve()

all_ds = load_from_disk(str(updated_data / 'processed/all'))
filtered_ds = load_from_disk(str(updated_data / 'processed/filtered'))
assert (len(all_ds['train']), len(all_ds['validation'])) == (469, 53), 'Unexpected D1 split; check Notebook 01.'
assert (len(filtered_ds['train']), len(filtered_ds['validation'])) == (367, 41), 'Unexpected D2 split; check Notebook 01.'

print('Data path:', local_data.resolve())
print('D1:', len(all_ds['train']), 'train /', len(all_ds['validation']), 'validation')
print('D2:', len(filtered_ds['train']), 'train /', len(filtered_ds['validation']), 'validation')
print('New checkpoints will be stored in:', updated_data / 'checkpoints')

Data path: /content/drive/MyDrive/SC4001/updated_data
D1: 469 train / 53 validation
D2: 367 train / 41 validation
New checkpoints will be stored in: /content/drive/MyDrive/SC4001/updated_data/checkpoints


In [5]:
# GPU check
import torch
assert torch.cuda.is_available(), 'No GPU — enable T4 via Runtime > Change runtime type'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU: Tesla T4 | VRAM: 15.6 GB


In [6]:
# Experiment 3 — train on all teacher traces (updated D1)
import yaml

with open('configs/finetune.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['mode'] = 'distillation'
cfg['data']['train_fraction'] = 1.0
cfg['training']['output_dir'] = 'data/checkpoints/distillation_updated'

checkpoint_dir = Path(cfg['training']['output_dir'])
assert checkpoint_dir.resolve() == (updated_data / 'checkpoints/distillation_updated').resolve()

with open('configs/finetune.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

# Save the exact configuration used, outside the Trainer output directory.
config_archive = updated_data / 'configs'
config_archive.mkdir(parents=True, exist_ok=True)
with (config_archive / 'finetune_exp3_updated.yaml').open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Experiment 3 | Mode: {cfg['data']['mode']} | Rank: {cfg['lora']['rank']} | Epochs: {cfg['training']['num_epochs']}")
print('Output:', checkpoint_dir.resolve())

from src.training.finetune import train

adapter_exists = any((checkpoint_dir / filename).is_file() for filename in ('adapter_model.safetensors', 'adapter_model.bin'))
config_exists = (checkpoint_dir / 'adapter_config.json').is_file()
if adapter_exists and config_exists:
    print('Exp 3 final adapter already exists — skipping training.')
elif adapter_exists or config_exists:
    raise RuntimeError('Exp 3 has an incomplete final adapter. Inspect the checkpoint folder before retraining.')
else:
    train()

Experiment 3 | Mode: distillation | Rank: 16 | Epochs: 3
Output: /content/drive/MyDrive/SC4001/updated_data/checkpoints/distillation_updated


W1115 12:19:08.228000 558 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1115 12:19:08.269000 558 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (14 outputs left out)

Building labels for eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/53 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
90,0.421292,0.365703,0.334332,0.896861,638934.000000


In [7]:
# Confirm Experiment 3 checkpoint before moving to Experiment 4
from pathlib import Path

exp3_dir = updated_data / 'checkpoints/distillation_updated'
assert exp3_dir.is_dir(), f'Experiment 3 output missing: {exp3_dir}'
assert any((exp3_dir / name).is_file() for name in ('adapter_model.safetensors', 'adapter_model.bin')), 'Experiment 3 final adapter not found'
assert (exp3_dir / 'adapter_config.json').is_file(), 'Experiment 3 adapter config missing'
print('Experiment 3 saved successfully:', exp3_dir)
print('Files:', sorted(p.name for p in exp3_dir.iterdir()))

# Free unused GPU cache before loading the student again.
import gc
gc.collect()
torch.cuda.empty_cache()

Experiment 3 saved successfully: /content/drive/MyDrive/SC4001/updated_data/checkpoints/distillation_updated
Files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'checkpoint-90', 'run_config.yaml', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


In [8]:
# Experiment 4 — train on correct-answer teacher traces (updated D2)
with open('configs/finetune.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['mode'] = 'filtered'
cfg['data']['train_fraction'] = 1.0
cfg['training']['output_dir'] = 'data/checkpoints/filtered_updated'

checkpoint_dir = Path(cfg['training']['output_dir'])
assert checkpoint_dir.resolve() == (updated_data / 'checkpoints/filtered_updated').resolve()

with open('configs/finetune.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

with (config_archive / 'finetune_exp4_updated.yaml').open('w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Experiment 4 | Mode: {cfg['data']['mode']} | Rank: {cfg['lora']['rank']} | Epochs: {cfg['training']['num_epochs']}")
print('Output:', checkpoint_dir.resolve())

adapter_exists = any((checkpoint_dir / filename).is_file() for filename in ('adapter_model.safetensors', 'adapter_model.bin'))
config_exists = (checkpoint_dir / 'adapter_config.json').is_file()
if adapter_exists and config_exists:
    print('Exp 4 final adapter already exists — skipping training.')
elif adapter_exists or config_exists:
    raise RuntimeError('Exp 4 has an incomplete final adapter. Inspect the checkpoint folder before retraining.')
else:
    train()

Experiment 4 | Mode: filtered | Rank: 16 | Epochs: 3
Output: /content/drive/MyDrive/SC4001/updated_data/checkpoints/filtered_updated


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Adding EOS to train dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

... (4 outputs left out)

Building labels for eval dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/41 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
69,0.397150,0.310875,0.282696,0.910008,484296.000000


In [9]:
# Final check — both updated adapters saved in Google Drive
for experiment, name in ((3, 'distillation_updated'), (4, 'filtered_updated')):
    folder = updated_data / 'checkpoints' / name
    adapter = any((folder / filename).is_file() for filename in ('adapter_model.safetensors', 'adapter_model.bin'))
    config = (folder / 'adapter_config.json').is_file()
    print(f'Experiment {experiment}:', folder)
    print('  Adapter saved:', adapter, '| Config saved:', config)
    assert adapter and config, f'Experiment {experiment} final checkpoint missing'

print('Both updated experiments are trained and saved to Google Drive.')
print('Share these two checkpoint folders with the teammate running Notebook 03.')

Experiment 3: /content/drive/MyDrive/SC4001/updated_data/checkpoints/distillation_updated
  Adapter saved: True | Config saved: True
Experiment 4: /content/drive/MyDrive/SC4001/updated_data/checkpoints/filtered_updated
  Adapter saved: True | Config saved: True
Both updated experiments are trained and saved to Google Drive.
Share these two checkpoint folders with the teammate running Notebook 03.
